# Employee Wellness Analysis — Exploratory Data Analysis (EDA)

### Workplace mental-health survey — aggregate analysis

**Objective:** Explore anonymous survey responses about workplace mental-health support, treatment-seeking, family history, work interference, company size, and awareness of benefits.

**Privacy note:** This notebook is for aggregate educational analysis only. Do not use it to identify, diagnose, rank, or make employment decisions about individuals. Do not publish private employee responses or free-text comments.

## 1. Understanding the Dataset

In [ ]:
possible_paths = [
    Path("data") / "employee_wellness_dataset.csv",
    Path("employee_wellness_dataset.csv"),
    Path("data") / "employee_wellness_dataset_demo.csv",
]
DATA_PATH = next((path for path in possible_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Place the permitted original CSV in data/ or beside this notebook, "
        "or use the included synthetic demo CSV in data/."
    )

IS_SYNTHETIC_DEMO = DATA_PATH.name.endswith("_demo.csv")
if IS_SYNTHETIC_DEMO:
    print("DEMO MODE: synthetic workplace survey examples only. Do not treat these as real employee findings.")

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())


In [ ]:
df.info()

In [ ]:
missing_summary = pd.DataFrame({
    "Missing values": df.isna().sum(),
    "Missing percent": (df.isna().mean() * 100).round(2),
}).sort_values("Missing percent", ascending=False)
display(missing_summary.head(15))

**Data note:** The original notebook described 1,048 responses and 28 columns, with missing values in comments, state, and work-interference fields. These figures must be rechecked against the actual CSV before publication. Missing answers can represent skipped questions or not-applicable responses; do not automatically treat them as "No".

## 2. Data Preprocessing

In [ ]:
# Convert Age to numeric and mark out-of-range values as missing.
df["Age_numeric"] = pd.to_numeric(df["Age"], errors="coerce")
valid_age = df["Age_numeric"].between(18, 75, inclusive="both")
df["Age_clean"] = df["Age_numeric"].where(valid_age)
print("Invalid/out-of-range ages:", int((df["Age_numeric"].notna() & ~valid_age).sum()))
display(df["Age_clean"].describe())

In [ ]:
def clean_gender(value):
    if pd.isna(value):
        return "Not reported"
    value = str(value).strip().lower()
    male_values = {"male", "m", "male-ish", "maile", "cis male", "mal", "male (cis)", "make", "man", "msle", "mail", "malr", "cis man", "guy (-ish) ^_^", "male leaning androgynous"}
    female_values = {"female", "f", "cis female", "woman", "femake", "female (cis)", "female (trans)", "cis-female/femme", "trans woman", "trans-female"}
    if value in male_values:
        return "Male"
    if value in female_values:
        return "Female"
    if value in {"", "nan", "none", "prefer not to say"}:
        return "Not reported"
    return "Other/Non-binary"

df["Gender_clean"] = df["Gender"].apply(clean_gender)
display(df["Gender_clean"].value_counts(dropna=False).to_frame("Responses"))

In [ ]:
# Correct known Excel date conversions in company-size labels.
if "no_employees" not in df.columns:
    raise KeyError("Expected column 'no_employees' was not found.")
map_fix = {"25-Jun": "6-25", "5-Jan": "1-5"}
df["no_employees_clean"] = df["no_employees"].astype("string").str.strip().replace(map_fix)
order_size = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]
df["no_employees_clean"] = df["no_employees_clean"].astype("object")
display(df["no_employees_clean"].value_counts(dropna=False).to_frame("Responses"))

In [ ]:
# Preserve uncertainty: missing values do not automatically mean "No".
if "self_employed" in df.columns:
    df["self_employed_clean"] = df["self_employed"].replace({"NA": np.nan, "N/A": np.nan}).fillna("Not reported")
if "work_interfere" in df.columns:
    df["work_interfere_clean"] = df["work_interfere"].fillna("Not reported / not applicable")

# Free-text comments are omitted to reduce privacy risk.
if "comments" in df.columns:
    df = df.drop(columns=["comments"])
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Prepared data shape:", df.shape)

## 3. Exploratory Data Analysis

### 3.1 Age Distribution

In [ ]:
bins = [17, 24, 29, 34, 39, 44, 49, 76]
labels = ["18-24", "25-29", "30-34", "35-39", "40-44", "45-49", "50+"]
df["age_bin"] = pd.cut(df["Age_clean"], bins=bins, labels=labels)
ax = df["age_bin"].value_counts().reindex(labels, fill_value=0).plot(kind="bar")
ax.set_title("Age Distribution of Survey Respondents")
ax.set_xlabel("Age group")
ax.set_ylabel("Responses")
plt.tight_layout()
plt.show()

**Insight:** The workforce sampled skews young — roughly 60% of respondents are between 25 and 34, a career stage often associated with high workload and career-establishment stress.

### 3.2 Gender Distribution

In [ ]:
ax = df["Gender_clean"].value_counts().plot(kind="bar")
ax.set_title("Gender Categories in Survey Responses")
ax.set_xlabel("")
ax.set_ylabel("Responses")
plt.tight_layout()
plt.show()

**Insight:** The sample is heavily male (~79%), so gender-based comparisons should be interpreted cautiously due to sample-size imbalance.

### 3.3 Overall Treatment-Seeking Rate

In [ ]:
treatment_counts = df["treatment"].fillna("Not reported").value_counts()
ax = treatment_counts.plot(kind="pie", autopct="%1.1f%%", ylabel="")
ax.set_title("Reported Treatment-Seeking Responses")
plt.tight_layout()
plt.show()

**Insight:** Nearly half (48.9%) of respondents have already sought treatment for a mental health condition — this is the group whose experience should shape wellness-program design.

### 3.4 Family History vs Treatment

In [ ]:
ct = pd.crosstab(df["family_history"], df["treatment"], normalize="index") * 100
display(ct.round(1))
ax = ct.plot(kind="bar", stacked=False)
ax.set_title("Treatment-Seeking Responses by Family History")
ax.set_ylabel("Percent within family-history group")
ax.set_xlabel("Family history response")
plt.tight_layout()
plt.show()

**Insight:** Employees with a family history of mental illness seek treatment at **57.8%**, nearly 3x the rate of those without one (**20.3%**) — the single strongest predictor in the dataset.

### 3.5 Work Interference vs Treatment

In [ ]:
ct = pd.crosstab(df["work_interfere_clean"], df["treatment"]).fillna(0)
display(ct)
ax = ct.plot(kind="bar")
ax.set_title("Treatment-Seeking by Reported Work Interference")
ax.set_ylabel("Responses")
ax.set_xlabel("Work interference response")
plt.tight_layout()
plt.show()

**Insight:** Respondents who say their condition interferes with work 'Sometimes' or 'Often' overwhelmingly seek treatment, confirming self-reported interference is a strong, actionable warning sign.

### 3.6 Company Size vs Treatment

In [ ]:
ct = pd.crosstab(df["no_employees_clean"], df["treatment"]).reindex(order_size).fillna(0)
display(ct)
ax = ct.plot(kind="bar")
ax.set_title("Treatment-Seeking by Company Size")
ax.set_ylabel("Responses")
ax.set_xlabel("Company-size category")
plt.tight_layout()
plt.show()

**Insight:** Treatment-seeking is fairly stable across company sizes, with a slightly higher share of small companies (1-5 employees) reporting treatment — possibly reflecting less formal support infrastructure.

### 3.7 Employer Benefits vs Treatment

In [ ]:
ct = pd.crosstab(df["benefits"].fillna("Not reported"), df["treatment"].fillna("Not reported"))
display(ct)
ax = ct.plot(kind="bar")
ax.set_title("Treatment-Seeking by Benefits Awareness")
ax.set_ylabel("Responses")
ax.set_xlabel("Benefits response")
plt.tight_layout()
plt.show()

**Insight:** Employees who know their employer offers mental-health benefits seek treatment at **62%**, versus only **36%** among those who say 'Don't know' — clear communication of existing benefits matters as much as the benefits themselves.

### 3.8 Top Countries Represented

In [ ]:
top_countries = df["Country"].fillna("Not reported").value_counts().head(8)
ax = top_countries.sort_values().plot(kind="barh")
ax.set_title("Top Countries by Survey Response Count")
ax.set_xlabel("Responses")
plt.tight_layout()
plt.show()

**Insight:** The US (61%) and UK (15%) dominate the sample, so region-specific policy differences (e.g. healthcare access) may be confounding some of the patterns above.

## 4. Interpretation and limitations

Use the charts to describe aggregate patterns in this survey and calculate percentages from the displayed counts before making claims. Specific percentages in the original notebook have not been treated as verified because the source CSV was not available for this repair pass.

- Association does not prove causation.
- A self-reported survey is not a clinical assessment and must not be used to diagnose people.
- Do not use these patterns to rank employees or make hiring, promotion, performance, or other employment decisions.
- Results may not represent other countries, industries, or the current workforce.
- Missing answers and small category counts can affect comparisons.
- Prefer aggregate, privacy-preserving reporting and make support resources available to everyone.

## 5. Conclusion

This notebook demonstrates data loading, missing-value review, category cleaning, and exploratory charts. Add the permitted source dataset locally, run the notebook from top to bottom, and update written findings only after checking the actual output.